Here we do an initial combination of the Open Air Chicago clarity sensors and satellite retrieved NO2 tropospheric column measurements.
Hourly and spatial aggregations on the open air data are performed. Basic statistic metrics are calculated. 
This notebook does not currently filter out pixels with single sites, but that is planned as a future addition per request of Carl Malings. 

- Authorship: Noah Ring
- Year: 2026
- Program: EarthRISE Developers Academy
- Location: Langley Research Center

In [ ]:
import xarray as xr
import matplotlib as plt
import pandas as pd
import geopandas as gpd
import numpy as np

# Change the matplotlib dpi to increase or decrease output size in vscode
plt.rcParams['figure.dpi'] = 150 

# Change some of the default xarray settings
xr.set_options(keep_attrs=True, display_expand_data=False)

# EPSG to be used when setting coordinate reference systems
epsg = 4326 

# Function used for coercing clarity lat/lon to match the 0.2 x 0.2deg grid of TEMPO
# Note: I included the bias term (0.01) to force the lat/lon to match TEMPO lat/lon
def round_002(x):
    out = 0.02*np.floor(x/0.02) + 0.01
    return np.round(out, 2)

write_files = True 

In [ ]:

#Define the path where the tempo.nc file is stored.
tempo_path = "~/Documents/01_Research/larc-chicago-haq/datasets/tempo.nc"

tempo_addvar_path = "~/Documents/01_Research/larc-chicago-haq/datasets/tempo_addVar.nc"

# Define the path where the open air chicago clarity data is stored
clarity_path = "~/Documents/01_Research/larc-chicago-haq/datasets/Open_Air_Chicago_Individual_Measurements_20260615.csv"


# Use xarray to load tempo data
tempo_raster = xr.open_dataset(tempo_path, decode_coords=True, decode_times=True)
tempo_raster = tempo_raster.rename({"vertical_column_troposphere":"column_no2"})
tempo_raster = tempo_raster.rename({"time": "datetime"})
tempo_pbl_rast = xr.open_dataset(tempo_addvar_path,decode_coords=True, decode_times=True)

tempo_pbl_rast = tempo_pbl_rast.pbl_height
tempo_pbl_rast = tempo_pbl_rast.rename({"time": "datetime"})

tempo_pbl = tempo_pbl_rast.to_dataframe().reset_index()
tempo_subset = tempo_raster.to_dataframe().reset_index()

tempo_pbl['latitude'] = tempo_pbl['latitude'].apply(lambda x: round_002(x))
tempo_pbl['longitude'] = tempo_pbl['longitude'].apply(lambda x: round_002(x))
tempo_subset['latitude'] = tempo_subset['latitude'].apply(lambda x: round_002(x))
tempo_subset['longitude'] = tempo_subset['longitude'].apply(lambda x: round_002(x))

tempo_subset = tempo_subset.merge(tempo_pbl, on=['latitude','longitude','datetime'], how="outer")
tempo_subset['hour'] = tempo_subset['datetime'].dt.hour
tempo_subset['doy'] = tempo_subset['datetime'].dt.dayofyear
tempo_subset['month'] = tempo_subset['datetime'].dt.month
tempo_subset["year"] = tempo_subset['datetime'].dt.year

# Load clarity data .csv
clarity_full_dat = pd.read_csv(clarity_path)

# Subset the clarity sensor data for our variables of interest
# Reformat the datetime variable, and add other time columns to possibly be used later
clarity_subset = clarity_full_dat[['datasourceid', 'sensor_name', 'latitude', 'longitude', 'time',
                            'no2ConcIndividual.value']]
clarity_subset['datetime'] = pd.to_datetime(clarity_subset["time"], format="%m/%d/%Y %I:%M:%S %p")
clarity_subset['hour'] = clarity_subset['datetime'].dt.hour
clarity_subset['doy'] = clarity_subset['datetime'].dt.dayofyear
clarity_subset['month'] = clarity_subset['datetime'].dt.month
clarity_subset["year"] = clarity_subset['datetime'].dt.year


start_date = tempo_subset['datetime'].min()
end_date = tempo_subset['datetime'].max()

date_mask = ( clarity_subset['datetime'] >= start_date ) & (clarity_subset['datetime'] <= end_dar
te)
clarity_subset = clarity_subset.loc[date_mask]

# Apply rounding function to the lat/lon columns of our subset clarity data
clarity_subset['latitude'] = clarity_subset['latitude'].apply(lambda x: round_002(x))
clarity_subset['longitude'] = clarity_subset['longitude'].apply(lambda x: round_002(x))


sensors_per_pixel = clarity_subset.groupby(['latitude','longitude']).agg(sensor_count=('datasourceid', 'nunique'))

In [ ]:
print("min tempo lat:", tempo_subset['latitude'].min(), "max tempo lat:",tempo_subset['latitude'].max())
print("min clarity lat:", clarity_subset['latitude'].min(), "max clarity lat:",clarity_subset['latitude'].max())
print("min tempo lon:", tempo_subset['longitude'].min(), "max tempo lon:",tempo_subset['longitude'].max())
print("min clarity lon:", clarity_subset['longitude'].min(), "max clarity lon:",clarity_subset['longitude'].max())
print(tempo_subset.head())
        # 
print(clarity_subset.head())        

### Function for combining data.
This requires the above cell to be run to execute properly.

In [ ]:
def combine_data(time_grouper, output_path, write_files, freq="1h"):

    if time_grouper == 'datetime' or time_grouper == 'time':
        frequency = freq
        group_var = pd.Grouper('datetime', freq=frequency)
    else: 
        group_var = time_grouper

    index = time_grouper

    clarity_subset_hour = (
    clarity_subset.set_index(index).groupby([group_var, 'latitude', 'longitude'])
    .agg( 
        sfc_no2_mean=('no2ConcIndividual.value', 'mean'),
        sfc_no2_median=('no2ConcIndividual.value', 'median'),
        sfc_no2_var=('no2ConcIndividual.value', 'var'),
        sfc_no2_sd=('no2ConcIndividual.value', 'std')
        ).reset_index()
    .merge(sensors_per_pixel, how="left", on=['latitude','longitude'])
    )

    tempo_subset_hour = (
    tempo_subset.set_index(index).groupby([group_var, 'latitude', 'longitude'])
        .agg( 
            col_no2_mean=('column_no2', 'mean'),
            col_no2_median=('column_no2', 'median'),
            col_no2_var=('column_no2', 'var'),
            col_no2_sd=('column_no2', 'std'),
            pbl_mean=('pbl_height', 'mean'),
            pbl_median=('pbl_height', 'median'),
            pbl_var=('pbl_height', 'var'),
            pbl_sd=('pbl_height', 'std'),
            ).reset_index()
    )

    clarity_tempo_hour = (
        clarity_subset_hour.merge(tempo_subset_hour, 
                                                on=[index,'latitude','longitude'],
                                                how='left',
                                                validate='many_to_one')
                                                )

    if write_files == True:
        clarity_tempo_hour.to_csv(output_path)

    print(clarity_tempo_hour)



In [ ]:
output_path = "~/Documents/01_Research/larc-chicago-haq/datasets/clarity-tempo-hourly.csv"

combine_data('datetime', freq = '1h', output_path=output_path, write_files=write_files)


In [ ]:
output_path = "~/Documents/01_Research/larc-chicago-haq/datasets/clarity-tempo-diurnal.csv"

combine_data(time_grouper='hour', output_path=output_path, write_files=write_files)


In [ ]:
# Group by time and lat/lon. This will put multiple stations at the same lat/lon coordinate, and aggregate their data together.
# The 'freq' option in pd.Grouper can be changed to 'h' for doing diel cycle avgs over the whole study period
clarity_subset_hour = (
    clarity_subset.set_index('datetime').groupby([pd.Grouper('datetime', freq='1h'), 'latitude', 'longitude'])
    .agg( 
        sfc_no2_mean=('no2ConcIndividual.value', 'mean'),
        sfc_no2_median=('no2ConcIndividual.value', 'median'),
        sfc_no2_var=('no2ConcIndividual.value', 'var'),
        sfc_no2_sd=('no2ConcIndividual.value', 'std')
        ).reset_index()
        .merge(sensors_per_pixel, how="left", on=['latitude','longitude']))


tempo_subset_hour = (
    tempo_subset.set_index('datetime').groupby([pd.Grouper('datetime', freq='1h'), 'latitude', 'longitude'])
    .agg( 
        col_no2_mean=('column_no2', 'mean'),
        col_no2_median=('column_no2', 'median'),
        col_no2_var=('column_no2', 'var'),
        col_no2_sd=('column_no2', 'std'),
        pbl_mean=('pbl_height', 'mean'),
        pbl_median=('pbl_height', 'median'),
        pbl_var=('pbl_height', 'var'),
        pbl_sd=('pbl_height', 'std'),
        ).reset_index()
)

clarity_tempo_hour = (
    clarity_subset_hour.merge(tempo_subset_hour, 
                                             on=['datetime','latitude','longitude'],
                                             how='left',
                                             validate='many_to_one')
                                             )

if write_files == True:
    output_path = "~/Documents/01_Research/larc-chicago-haq/datasets/clarity-tempo-hourly.csv"
    clarity_tempo_hour.to_csv(output_path)

# tempo_hour_mean = tempo_raster.column_no2.resample(time='1h').mean(dim='time')
# tempo_hour_median = tempo_raster.column_no2.resample(time='1h').median(dim='time')
# tempo_hour_sd = tempo_raster.column_no2.resample(time='1h').std(dim='time')
# tempo_hour_var = tempo_raster.column_no2.resample(time='1h').var(dim='time')

# # Combine the above into a single dataframe.
# tempo_hour_stats = xr.merge([
#     tempo_hour_mean.rename('column_no2_mean'),
#     tempo_hour_median.rename('column_no2_median'),
#     tempo_hour_sd.rename('column_no2_sd'),
#     tempo_hour_var.rename('column_no2_var')
# ])

# # Rename the clarity datetime column to match the tempo time column
# clarity_tempo_hour = clarity_subset_hour.rename(columns={'datetime': 'time'})
# # Define the index as spatial and temporal prior to converting to xarray
# clarity_tempo_hour = clarity_tempo_hour.set_index(['time', 'latitude', 'longitude']).to_xarray()

# # Normalize coord precision before merging so matching is consistent.
# # This step is kind of hacky but necessary to ensure coordinates match exactly between clarity and tempo
# # If there is a better way to do this, then please replace this. Without this step you get NaN values when combining.
# # clarity_tempo_hour = clarity_tempo_hour.assign_coords(
# #     latitude=((clarity_tempo_hour.latitude.astype('float32') * 100).astype('int32') / 100).astype('float32'),
# #     longitude=((clarity_tempo_hour.longitude.astype('float32') * 100).astype('int32') / 100).astype('float32'),
# # )

# # tempo_hour_stats = tempo_hour_stats.assign_coords(
# #     latitude=((tempo_hour_stats.latitude.astype('float32') * 100).astype('int32') / 100).astype('float32'),
# #     longitude=((tempo_hour_stats.longitude.astype('float32') * 100).astype('int32') / 100).astype('float32'),
# # )

# # Here, we right join the clarity dataset to the tempo dataset.
# # This keeps all points in tempo, but only matching points from clarity 
# clarity_tempo_hour = xr.merge([clarity_tempo_hour, tempo_hour_stats], join='right', compat='no_conflicts')

# clarity_tempo_hour

# if write_files == True:
#     output_path = "~/Documents/01_Research/larc-chicago-haq/datasets/clarity-tempo-hourly.nc"
#     clarity_tempo_hour.to_netcdf(output_path)

In [ ]:
clarity_tempo_hour

### Combining TEMPO and Clarity - Diurnal

In [ ]:
clarity_subset_diel = clarity_subset.copy()
clarity_subset_diel['hour'] = clarity_subset_diel['datetime'].dt.hour

clarity_subset_diel = (
    clarity_subset_diel.set_index('hour').groupby(['hour', 'latitude', 'longitude'])
    .agg( 
        sfc_no2_mean=('no2ConcIndividual.value', 'mean'),
        sfc_no2_median=('no2ConcIndividual.value', 'median'),
        sfc_no2_var=('no2ConcIndividual.value', 'var'),
        sfc_no2_sd=('no2ConcIndividual.value', 'std')
        ).reset_index()
        .merge(sensors_per_pixel, how="left", on=['latitude','longitude']))


tempo_subset_diel = tempo_subset.copy()
tempo_subset_diel['hour'] = tempo_subset_diel['datetime'].dt.hour

tempo_subset_diel = (
    tempo_subset_diel.set_index('hour').groupby(['hour', 'latitude', 'longitude'])
    .agg( 
        col_no2_mean=('column_no2', 'mean'),
        col_no2_median=('column_no2', 'median'),
        col_no2_var=('column_no2', 'var'),
        col_no2_sd=('column_no2', 'std')
        ).reset_index()
)

clarity_tempo_diel = (
    clarity_subset_diel.merge(tempo_subset_diel, 
                                             on=['hour','latitude','longitude'],
                                             how='left',
                                             validate='many_to_one')
                                             )

if write_files == True:
    output_path = "~/Documents/01_Research/larc-chicago-haq/datasets/clarity-tempo-diurnal.csv"
    clarity_tempo_diel.to_csv(output_path)



# clarity_subset_diel = (
#     clarity_subset_diel.groupby(['hour', 'latitude', 'longitude'])
#     .agg(
#         sfc_no2_mean=('no2ConcIndividual.value', 'mean'),
#         sfc_no2_median=('no2ConcIndividual.value', 'median'),
#         sfc_no2_var=('no2ConcIndividual.value', 'var'),
#         sfc_no2_sd=('no2ConcIndividual.value', 'std')
#     )
#     .reset_index()
#     .merge(sensors_per_pixel, how="left", on=['latitude','longitude'])
# )

# # Calculate the diurnal cycle for TEMPO by aggregating the same hour across all days.
# tempo_diel_mean = tempo_raster.column_no2.groupby('time.hour').mean(dim='time')
# tempo_diel_median = tempo_raster.column_no2.groupby('time.hour').median(dim='time')
# tempo_diel_sd = tempo_raster.column_no2.groupby('time.hour').std(dim='time')
# tempo_diel_var = tempo_raster.column_no2.groupby('time.hour').var(dim='time')

# # Combine the above into a single dataframe.
# tempo_diel_stats = xr.merge([
#     tempo_diel_mean.rename('column_no2_mean'),
#     tempo_diel_median.rename('column_no2_median'),
#     tempo_diel_sd.rename('column_no2_sd'),
#     tempo_diel_var.rename('column_no2_var')
# ])

# # Rename the clarity hour column to match the TEMPO hour coordinate.
# clarity_tempo_diel = clarity_subset_diel
# # Define the index as spatial and temporal prior to converting to xarray
# clarity_tempo_diel = clarity_tempo_diel.set_index(['hour', 'latitude', 'longitude']).to_xarray()

# # Normalize coord precision before merging so matching is consistent.
# # This step is kind of hacky but necessary to ensure coordinates match exactly between clarity and tempo
# # If there is a better way to do this, then please replace this. Without this step you get NaN values when combining.


# # Here, we right join the clarity dataset to the tempo dataset.
# # This keeps all points in tempo, but only matching points from clarity 
# clarity_tempo_diel = xr.merge([clarity_tempo_diel, tempo_diel_stats], join='outer', compat='no_conflict')

# if write_files==True:
#     output_path = "~/Documents/01_Research/larc-chicago-haq/datasets/clarity-tempo-diurnal.nc"
#     clarity_tempo_diel.to_netcdf(output_path)



In [ ]:
clarity_tempo_diel

## Combine Clarity and TEMPO - Filtered Months - Time of Day aggregation
We are filtering out months with large amounts of missing data (Dec 2025, Jan 2026, Feb 2026) then doing the hourly time of day aggregation. Since we only have less than a year of data, I am filtering by numeric rather than datetime. Ideally it should be filtered by datetime. 

In [ ]:
month_filter = [12,1,2]


clarity_tod_month = clarity_subset.query('month not in @month_filter')
clarity_tod_month['hour'] = clarity_tod_month['datetime'].dt.hour

clarity_tod_month = (
    clarity_tod_month.set_index('hour').groupby(['hour', 'latitude', 'longitude'])
    .agg( 
        sfc_no2_mean=('no2ConcIndividual.value', 'mean'),
        sfc_no2_median=('no2ConcIndividual.value', 'median'),
        sfc_no2_var=('no2ConcIndividual.value', 'var'),
        sfc_no2_sd=('no2ConcIndividual.value', 'std')
        ).reset_index()
        .merge(sensors_per_pixel, how="left", on=['latitude','longitude']))

tempo_tod_month = tempo_subset
tempo_tod_month['month'] = tempo_tod_month['datetime'].dt.month
tempo_tod_month['hour'] = tempo_tod_month['datetime'].dt.hour
tempo_tod_month = tempo_tod_month.query('month not in @month_filter')

tempo_tod_month = (
    tempo_tod_month.set_index('hour').groupby(['hour', 'latitude', 'longitude'])
    .agg( 
        col_no2_mean=('column_no2', 'mean'),
        col_no2_median=('column_no2', 'median'),
        col_no2_var=('column_no2', 'var'),
        col_no2_sd=('column_no2', 'std')
        ).reset_index()
)

clarity_tempo_nowinter = (
    clarity_tod_month.merge(tempo_tod_month, 
                                             on=['hour','latitude','longitude'],
                                             how='left',
                                             validate='many_to_one')
                                             )

if write_files == True:
    output_path = "~/Documents/01_Research/larc-chicago-haq/datasets/clarity-tempo-diurnal-nowinter.csv"
    clarity_tempo_nowinter.to_csv(output_path)


# print(np.sort(clarity_tod_month['month'].unique()))

# clarity_tod_month = (
#     clarity_tod_month.groupby(['hour', 'latitude', 'longitude'])
#     .agg(
#         sfc_no2_mean=('no2ConcIndividual.value', 'mean'),
#         sfc_no2_median=('no2ConcIndividual.value', 'median'),
#         sfc_no2_var=('no2ConcIndividual.value', 'var'),
#         sfc_no2_sd=('no2ConcIndividual.value', 'std')
#     )
#     .reset_index()
#     .merge(sensors_per_pixel, how="left", on=['latitude','longitude'])
# )

# Calculate the diurnal cycle for TEMPO by aggregating the same hour across all days.
# tempo_raster_filtered = tempo_raster.where(~tempo_raster.time.dt.month.isin(month_filter), drop=True) 



# tempo_diel_mean = tempo_raster_filtered.column_no2.groupby('time.hour').mean(dim='time')
# tempo_diel_median = tempo_raster_filtered.column_no2.groupby('time.hour').median(dim='time')
# tempo_diel_sd = tempo_raster_filtered.column_no2.groupby('time.hour').std(dim='time')
# tempo_diel_var = tempo_raster_filtered.column_no2.groupby('time.hour').var(dim='time')

# # Combine the above into a single dataframe.
# tempo_diel_stats = xr.merge([
#     tempo_diel_mean.rename('column_no2_mean'),
#     tempo_diel_median.rename('column_no2_median'),
#     tempo_diel_sd.rename('column_no2_sd'),
#     tempo_diel_var.rename('column_no2_var')
# ])

# # Rename the clarity hour column to match the TEMPO hour coordinate.
# combined_tod_month =clarity_tod_month 
# # Define the index as spatial and temporal prior to converting to xarray
# combined_tod_month = combined_tod_month.set_index(['hour', 'latitude', 'longitude']).to_xarray()

# Normalize coord precision before merging so matching is consistent.
# This step is kind of hacky but necessary to ensure coordinates match exactly between clarity and tempo
# If there is a better way to do this, then please replace this. Without this step you get NaN values when combining.

# Here, we right join the clarity dataset to the tempo dataset.
# This keeps all points in tempo, but only matching points from clarity 
# combined_tod_month = xr.merge([combined_tod_month, tempo_diel_stats], join='right', compat='override')

# print(np.unique(tempo_raster_filtered.time.dt.month.values))

# if write_files==True:
#     output_path = "~/Documents/01_Research/larc-chicago-haq/datasets/clarity-tempo-nowinter-diurnal.nc"
#     combined_tod_month.to_netcdf(output_path)



# Combine Clarity & Tempo - No Averaging / Aggregation

In [ ]:
clarity_dat = clarity_subset

clarity_dat_subset = clarity_dat[['year', 'month', 'doy', 'hour', 'datetime', 'latitude', 'longitude', 'no2ConcIndividual.value']]

tempo_no2 = tempo_subset[['column_no2','datetime','latitude','longitude']]
tempo_no2['hour'] = tempo_no2['datetime'].dt.hour
tempo_no2['doy'] = tempo_no2['datetime'].dt.dayofyear
tempo_no2['month'] = tempo_no2['datetime'].dt.month
tempo_no2["year"] = tempo_no2['datetime'].dt.year

# clarity_dat_subset['latitude'] = ((clarity_dat_subset['latitude'].astype('float32') * 100).astype('int32') / 100).astype('float32')
# clarity_dat_subset['longitude'] = ((clarity_dat_subset['longitude'].astype('float32') * 100).astype('int32') / 100).astype('float32')

# tempo_no2['latitude'] = ((tempo_no2['latitude'].astype('float32') * 100).astype('int32') / 100).astype('float32')
# tempo_no2['longitude'] = ((tempo_no2['longitude'].astype('float32') * 100).astype('int32') / 100).astype('float32')

combined = clarity_dat_subset.merge(tempo_no2, 
                                    on=['year', 'month', 'doy', 'hour', 'latitude','longitude'],
                                    how='outer',
                                    validate='many_to_many')


combined = combined.merge(sensors_per_pixel, how='left', on=['latitude','longitude'])

if write_files == True:
    output_path = "~/Documents/01_Research/larc-chicago-haq/datasets/clarity-tempo-full.csv"
    combined.to_csv(output_path)


In [ ]:
print(tempo_subset.info())

print(clarity_subset.info())

In [ ]:
clarity_dt = sorted(clarity_subset['datetime'])
print(clarity_dt)

# tempo_subset['datetime'] = tempo_subset['datetime'].dt.strftime("%Y-%m-%d %H:%M:%S")
tempo_dt = sorted(tempo_subset['datetime'])

print(tempo_dt)

# TESTING BELOW HERE

Important: 
- Before we can aggregate clarity sensor data over days / weeks, we need to filter out nightime values consistent with TEMPO
    - Might be able to combine prior to aggregating? then it should just inherit the time with tempo data availabiliy

In [ ]:


clarity_subset_daily = (
    clarity_subset.set_index('datetime').groupby([pd.Grouper('datetime', freq='1D'), 'latitude', 'longitude'])
    .agg( 
        sfc_no2_mean=('no2ConcIndividual.value', 'mean'),
        sfc_no2_median=('no2ConcIndividual.value', 'median'),
        sfc_no2_var=('no2ConcIndividual.value', 'var'),
        sfc_no2_sd=('no2ConcIndividual.value', 'std'),
        sensor_count=('datasourceid', 'nunique')
        ).reset_index()
)


clarity_subset_all_days = (
    clarity_subset.set_index('datetime').groupby([pd.Grouper('datetime', freq='1D'), 'latitude', 'longitude'])
    .agg( 
        sfc_no2_mean=('no2ConcIndividual.value', 'mean'),
        sfc_no2_median=('no2ConcIndividual.value', 'median'),
        sfc_no2_var=('no2ConcIndividual.value', 'var'),
        sfc_no2_sd=('no2ConcIndividual.value', 'std'),
        sensor_count=('datasourceid', 'nunique')
        ).reset_index()
)

In [ ]:

print(clarity_subset_hour)

count = (
    clarity_subset.groupby(['latitude', 'longitude'])
    .agg(num_sensors_per_pixel=('datasourceid', 'nunique'),
         num_sensors_per_pixel_2=('sensor_name', 'nunique'))
    .reset_index()
    .groupby('num_sensors_per_pixel')
    .agg(
        count=('num_sensors_per_pixel', 'count'),
        count2 = ('num_sensors_per_pixel_2', 'count')
    )
         )

count
